In [5]:
from json import load

from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [15]:
message = [
	{"role" : "system", "content": "you are a helpful assistant who answers questions about the world."},
	{"role" : "user", "content": "What is the HCMUT?"}
]

In [16]:
response = client.chat.completions.create(
	model="llama-3.1-8b-instant",
	messages=message,
)

In [17]:
print(response.choices[0].message.content)

The HCMUT likely refers to Ho Chi Minh City University of Technology (also known as Việt Nam National University, Ho Chi Minh City - University of Technology). It is a prestigious engineering university located in Ho Chi Minh City, Vietnam. If there's anything else you'd like to know about the HCMUT, please let me know.


In [18]:
def messages_for(website, system_prompt:str, user_prompt_prefix:str):
    return [
		{"role" : "system", "content": system_prompt},
		{"role" : "user", "content": user_prompt_prefix + website}
	]

model = "llama-3.1-8b-instant"

In [26]:
system_prompt = "you are a helpful assistant who can analyze the content of a website and answer questions about it and provide a short summary, ignore any questions that are not related to the content of the website. Respond in the following format: 1. summary of the website, 2. answer to the question if it is related to the content of the website, otherwise respond with 'I am sorry, I cannot answer that question because it is not related to the content of the website. Response in Markdown format. Do not wrap the markdown in a code block - respond with pure markdown.'"
user_prompt_prefix = """
Here are the contents of a website. Provide a short summary of this website. If it includes news or annoucements, then summarize these too.

"""

In [27]:
messages_for("https://kelvindikhang.id.vn/", system_prompt, user_prompt_prefix)

[{'role': 'system',
  'content': "you are a helpful assistant who can analyze the content of a website and answer questions about it and provide a short summary, ignore any questions that are not related to the content of the website. Respond in the following format: 1. summary of the website, 2. answer to the question if it is related to the content of the website, otherwise respond with 'I am sorry, I cannot answer that question because it is not related to the content of the website. Response in Markdown format. Do not wrap the markdown in a code block - respond with pure markdown.'"},
 {'role': 'user',
  'content': '\nHere are the contents of a website. Provide a short summary of this website. If it includes news or annoucements, then summarize these too.\n\nhttps://kelvindikhang.id.vn/'}]

In [28]:
import requests
from bs4 import BeautifulSoup
def fetch_website_content(url):
    try:
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        for script_or_style in soup(["script", "style", "header", "footer", "nav"]):
            script_or_style.decompose()
            
        return soup.get_text(separator=' ', strip=True)
    except Exception as e:
        return f"Error: {e}"

In [29]:
def summarize(url:str):
    website = fetch_website_content(url)
    response = client.chat.completions.create(
        model = "llama-3.1-8b-instant",
        messages = messages_for(website, system_prompt, user_prompt_prefix)
    )
    return response.choices[0].message.content

In [30]:
summarize("https://kelvindikhang.id.vn/")

'1. *Summary of the website:* This is a personal website of an AI engineer, Khang Bui Tran Duy, who is actively seeking opportunities in his field. The website showcases his skills, experience, and qualifications, inviting potential employers or collaborators to explore his work and download his CV.\n\n2. I am sorry, I cannot answer that question because it is not related to the content of the website.'

In [37]:
import asyncio
from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
from groq import Groq

async def fetch_and_click_website(url):
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        try:
            await page.goto(url, wait_until="networkidle")
            button_selector = "text='Explore My Work →'"
            if await page.locator(button_selector).is_visible():
                print("🚀 Đã tìm thấy nút, đang thực hiện click...")
                await page.locator(button_selector).click()
                await page.wait_for_load_state("networkidle")
                await asyncio.sleep(2) 
            else:
                print("⚠️ Không tìm thấy nút, lấy nội dung trang chủ.")
            full_html = await page.content()
            await browser.close()
            soup = BeautifulSoup(full_html, 'html.parser')
            for element in soup(["script", "style", "nav", "footer", "header", "aside", "svg"]):
                element.decompose()

            return soup.get_text(separator=' ', strip=True)

        except Exception as e:
            await browser.close()
            return f"Error: {str(e)}"

async def agent_summarize(url: str):
    website_content = await fetch_and_click_website(url)
    
    if "Error:" in website_content:
        return website_content

    messages = [
        {"role": "system", "content": "Bạn là chuyên gia phân tích portfolio AI Engineer."},
        {"role": "user", "content": f"Tóm tắt chi tiết dự án và kỹ năng từ nội dung sau:\n\n{website_content[:10000]}"}
    ]

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=messages
    )
    return response.choices[0].message.content

url = "https://kelvindikhang.id.vn/"
result = await agent_summarize(url)
print(result)


TargetClosedError: BrowserType.launch: Target page, context or browser has been closed
Browser logs:

<launching> /home/kelvindikhang/.cache/ms-playwright/chromium_headless_shell-1217/chrome-headless-shell-linux64/chrome-headless-shell --disable-field-trial-config --disable-background-networking --disable-background-timer-throttling --disable-backgrounding-occluded-windows --disable-back-forward-cache --disable-breakpad --disable-client-side-phishing-detection --disable-component-extensions-with-background-pages --disable-component-update --no-default-browser-check --disable-default-apps --disable-dev-shm-usage --disable-extensions --disable-features=AvoidUnnecessaryBeforeUnloadCheckSync,BoundaryEventDispatchTracksNodeRemoval,DestroyProfileOnBrowserClose,DialMediaRouteProvider,GlobalMediaControls,HttpsUpgrades,LensOverlay,MediaRouter,PaintHolding,ThirdPartyStoragePartitioning,Translate,AutoDeElevate,RenderDocument,OptimizationHints --enable-features=CDPScreenshotNewSurface --allow-pre-